# 07 — MedCPT Cross-Encoder Reranking (M6)

Runs `scripts/run_reranker.py`: reranks the hybrid RRF run's top candidates with `ncbi/MedCPT-Cross-Encoder`, sweeping candidate pool sizes {20, 50, 100} (ablation A6); pool=50 is the project's default/primary result.

**⚠️ Slowest step in this study:** ~765–3918 ms/query depending on pool size, measured on Apple M1 Pro (MPS) — all three pools together took ~40 minutes on that hardware. A Colab GPU should be substantially faster; consider running only the default pool for a quick check (see the config's `candidate_pool_sizes`).

**Prerequisite:** requires `results/runs/hybrid_rrf.trec` (notebook 06).

In [ ]:
# If running on Colab, clone the repo and install deps. Skipped automatically
# when already inside a local checkout (REPO_ROOT / 'src' already importable).
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('biomedical-hybrid-ir'):
        subprocess.run(['git', 'clone', 'https://github.com/Arungharami/biomedical-hybrid-ir'], check=True)
    os.chdir('biomedical-hybrid-ir')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('cwd:', os.getcwd())

In [ ]:
import time
start = time.perf_counter()
result = subprocess.run([sys.executable, 'scripts/run_reranker.py'], cwd=os.getcwd())
print(f'\nexit code: {result.returncode}, elapsed: {time.perf_counter()-start:.1f}s')
assert result.returncode == 0, 'Script failed -- see output above.'

## Real results (default pool=50)

In [ ]:
import json
payload = json.load(open('results/metrics/hybrid_reranked.json'))
print(json.dumps(payload, indent=2)[:3000])

## Candidate pool ablation (A6)

In [ ]:
import json
payload = json.load(open('results/metrics/reranker_pool_ablation.json'))
print(json.dumps(payload, indent=2)[:3000])

**Important:** Recall@100/MAP for the reranked run are capped by the candidate pool — a cross-encoder can only reorder candidates it's given. Verified exactly: pool=100's Recall@100 matches hybrid RRF's own Recall@100 to five decimal places. See `docs/models.md` §M6 for the full mechanism.

**M7 finding:** the nDCG@10 improvement at pool=50 (the best point estimate across all six models) is **not** statistically significant (p=0.194, n=323). P@10 *does* improve significantly (p=0.015).